# Review marked sites

Gathers **every landslide-candidate GeoJSON you've saved** (from any notebook, any location) onto one map for review, with the same BEFORE / TOPO views to re-inspect each spot. Re-run cell 1 anytime to pick up newly saved files.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA = (Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent) / "data"
files = sorted((DATA / "landslide_candidates").glob("*.geojson"))

frames = []
for f in files:
    gdf = gpd.read_file(f)
    gdf["source"] = f.stem
    frames.append(gdf)
sites = pd.concat(frames, ignore_index=True) if frames else None

if sites is None:
    print("No saved candidates yet — draw and export in notebooks 01/02/03 first.")
else:
    summary = sites.groupby("source").agg(
        features=("geometry", "size"),
        types=("geometry", lambda s: dict(s.geom_type.value_counts())),
    )
    print(f"{len(sites)} features from {len(files)} file(s)")
summary

44 features from 1 file(s)


,features,types
source,,
la-guaira_20260626_150535_17_3010,44,"{'Point': 26, 'Polygon': 18}"


## All sites on one map

- **Yellow** = your marked candidates (hover shows which file each came from)
- Switch BEFORE / TOPO with the pinned buttons to re-inspect the terrain under each mark
- Post-event scene footprints are outlined red so you can see which AFTER scene covers each site
  (open notebook 01 with that location to view it)

In [2]:
import leafmap

from geer_venezuela import (
    HILLSHADE,
    HILLSHADE_ATTRIBUTION,
    WAYBACK_ATTRIBUTION,
    WAYBACK_PRE_EVENT,
    add_compare_control,
    load_items,
)

m = leafmap.Map()
m.add_tile_layer(HILLSHADE, name="TOPO — terrain hillshade", attribution=HILLSHADE_ATTRIBUTION)
m.add_tile_layer(
    WAYBACK_PRE_EVENT,
    name="BEFORE — Esri Wayback 2026-05-28",
    attribution=WAYBACK_ATTRIBUTION,
    max_zoom=19,
)
footprints = load_items("post-event")[["id", "title", "location", "geometry"]]
m.add_gdf(
    footprints,
    layer_name="Post-event scene footprints",
    style={"color": "#ff3b30", "weight": 1.5, "fillOpacity": 0},
    zoom_to_layer=False,
)
m.add_gdf(
    sites,
    layer_name="Marked candidates",
    style={"color": "#ffd60a", "weight": 3, "fillOpacity": 0.4},
    zoom_to_layer=True,
)
add_compare_control(m, {"BEFORE": "BEFORE — Esri Wayback 2026-05-28", "TOPO": "TOPO — terrain hillshade"}, selected="BEFORE")
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## Site list with coordinates and BEFORE-image dates

One row per marked feature (polygons listed by centroid), with the actual capture date of the
pre-event imagery under each site — ready to paste into recommendations. The capture-date
lookup makes one web request per site, so this takes a few seconds.

In [3]:
from geer_venezuela import wayback_capture_date

table = sites.copy()
points = table.geometry.centroid
table["lat"] = points.y.round(5)
table["lon"] = points.x.round(5)
table["type"] = table.geometry.geom_type

before = [wayback_capture_date(lat, lon) for lat, lon in zip(table["lat"], table["lon"])]
table["before_date"] = [b["captured"] for b in before]
table["before_res_m"] = [b["resolution_m"] for b in before]

site_list = table[["source", "type", "lat", "lon", "before_date", "before_res_m"]]
site_list.to_csv(DATA / "landslide_candidates" / "site_list.csv", index_label="site_no")
print(f"Saved {DATA / 'landslide_candidates' / 'site_list.csv'}")
site_list

Saved /Users/lornearnold/GitHub/GEER_Venezuela/data/landslide_candidates/site_list.csv


,source,type,lat,lon,before_date,before_res_m
0,la-guaira_20260626_150535_17_3010,Point,10.54902,-66.98999,2025-02-20,0.34
1,la-guaira_20260626_150535_17_3010,Point,10.54354,-66.98793,2025-03-02,0.50
2,la-guaira_20260626_150535_17_3010,Point,10.53676,-66.98143,2025-03-02,0.50
3,la-guaira_20260626_150535_17_3010,Point,10.53385,-66.98363,2025-03-02,0.50
4,la-guaira_20260626_150535_17_3010,Point,10.52622,-66.99678,2025-03-02,0.50
5,la-guaira_20260626_150535_17_3010,Point,10.52639,-66.99422,2025-03-02,0.50
6,la-guaira_20260626_150535_17_3010,Point,10.52461,-66.99927,2025-03-02,0.50
7,la-guaira_20260626_150535_17_3010,Point,10.52112,-67.00239,2025-03-02,0.50
8,la-guaira_20260626_150535_17_3010,Polygon,10.52052,-66.99532,2025-03-02,0.50
9,la-guaira_20260626_150535_17_3010,Polygon,10.49090,-67.01972,2025-03-02,0.50


## Other saved products

Everything else you've exported lives next to the candidates and drags straight into QGIS/ArcGIS:

```
data/landslide_candidates/   your marked sites (+ site_list.csv from above)
data/routes/                 watch segments, arterials, corridor summary
data/terrain/                steep-area polygons, geology
data/usgs/                   USGS rapid assessment grid
```